# Three more examples

Run the two setup cells, and after that each section stands on its own. There are four checks on this page and the public runner allows six a minute, so running everything at once is fine — but if you do get rate limited, wait a moment and re-run the cell.

New to this? Start with [**`tla-in-your-browser.ipynb`**](/lab/index.html?path=tla-in-your-browser.ipynb) instead.

In [ ]:
%pip install -q tlakit

In [ ]:
%load_ext tlakit

import tlakit
from tlakit.magics import MODULES

## 1. A deadlock that is actually a deadlock

Two threads want the same two locks, and they disagree about the order. `a` takes `L1` then `L2`; `b` takes `L2` then `L1`. Each thread loops forever: take both, release both, start again — so a behaviour that runs out of steps has genuinely gone wrong, not merely finished.

In [ ]:
---- MODULE LockOrder ----
EXTENDS Naturals
VARIABLES owner, pc

Threads == {"a", "b"}
Locks == {"L1", "L2"}

\* Each thread wants both locks, and they disagree about the order.
Want == [a |-> <<"L1", "L2">>, b |-> <<"L2", "L1">>]

Init ==
  /\ owner = [l \in Locks |-> "free"]
  /\ pc = [t \in Threads |-> 1]

\* pc 1 and 2 are "waiting for my first / second lock"; 3 is "holding both".
Take(t) ==
  /\ pc[t] \in {1, 2}
  /\ owner[Want[t][pc[t]]] = "free"
  /\ owner' = [owner EXCEPT ![Want[t][pc[t]]] = t]
  /\ pc' = [pc EXCEPT ![t] = pc[t] + 1]

Release(t) ==
  /\ pc[t] = 3
  /\ owner' = [l \in Locks |-> IF owner[l] = t THEN "free" ELSE owner[l]]
  /\ pc' = [pc EXCEPT ![t] = 1]

Next == \E t \in Threads : Take(t) \/ Release(t)
Spec == Init /\ [][Next]_<<owner, pc>>
====

In [ ]:
%tlc:LockOrder
SPECIFICATION Spec

Two steps is all it takes. `a` holds `L1` and wants `L2`; `b` holds `L2` and wants `L1`; neither `Take` is enabled and neither thread is at `pc = 3`, so nothing can `Release`. Nobody moves again.

This is the case where the deadlock report means what it says — contrast it with `Atomic` in the tutorial, which trips the same check simply by finishing.

## 2. The same spec as a state graph

The whole reachable state space here is six states, which is small enough to read directly. Ask for it with `graph=True`.

In [ ]:
lock_order = tlakit.Spec(source=MODULES["LockOrder"], name="LockOrder")

# CHECK_DEADLOCK FALSE so TLC explores the whole space instead of stopping at
# the first dead end -- the graph should show every state, not a prefix.
result = lock_order.check(graph=True, check_deadlock=False)

print(result.outcome.value, "--", len(result.graph.nodes), "states,",
      len(result.graph.edges), "transitions")

# A deadlock is a node with no way out. Find them structurally.
has_exit = {edge.source for edge in result.graph.edges}
for node in result.graph.nodes:
    stuck = "   <-- no successor" if node.id not in has_exit else ""
    print(f"{node.variables['owner']:<30} {node.variables['pc']}{stuck}")

Turning the deadlock check off did not make the dead end go away — it is still right there in the graph, as the one node nothing leaves. The check is what turns that piece of structure into a report you can act on.

## 3. A property about the future

Invariants say what must be true of every state. Some things worth promising are not like that: *the retry eventually succeeds* is a claim about whole behaviours, and no single state can violate it. This one is checked from Python rather than a config cell — same check, and it leaves the result in a variable to pick apart afterwards.

In [ ]:
---- MODULE Retry ----
VARIABLE state

Init == state = "idle"

Attempt == state = "idle"   /\ state' = "trying"
GiveUp  == state = "trying" /\ state' = "idle"
Succeed == state = "trying" /\ state' = "done"

Next == Attempt \/ GiveUp \/ Succeed

\* Weak fairness on Next only promises that *something* keeps happening.
Spec     == Init /\ [][Next]_state /\ WF_state(Next)

\* Strong fairness on Succeed promises that an action enabled infinitely
\* often eventually happens.
FairSpec == Init /\ [][Next]_state /\ WF_state(Next) /\ SF_state(Succeed)

Finishes == <>(state = "done")
====

In [ ]:
retry = tlakit.Spec(source=MODULES["Retry"], name="Retry")

# check_deadlock=False for the same reason as above: `done` is a terminal
# state, and being told about it would bury the answer we came for.
bad = retry.check(properties=["Finishes"], check_deadlock=False)
bad

The counterexample is a **lasso**: a run into a cycle the behaviour then repeats forever. Here the cycle is `idle → trying → idle → …`, retrying and giving up, never succeeding. `WF_state(Next)` did not rule it out, because it only promises that *some* step keeps being taken — and these steps qualify.

A counterexample of infinite length still arrives as ordinary data. The trace splits at `loop_start`: everything before it happens once, everything from it onwards repeats forever. This particular loop closes on the very first state, so the one-time prefix is empty.

In [ ]:
print("outcome:   ", bad.outcome.value)
print("is lasso:  ", bad.trace.is_lasso)
print("loop_start:", bad.trace.loop_start)
print("happens once:", [s["state"] for s in bad.trace.prefix] or "(nothing)")
print("then forever:", [s["state"] for s in bad.trace.loop])

The fix is to promise more. `SF_state(Succeed)` — strong fairness — says that an action enabled infinitely often must eventually be taken. `Succeed` is enabled every time the spec passes through `trying`, so the retry loop stops being a legal behaviour.

Weak fairness would not have been enough: it only applies to an action that stays *continuously* enabled, and `Succeed` switches off every time the spec returns to `idle`. That distinction is the whole reason TLA⁺ has two fairness operators.

In [ ]:
good = retry.check(spec="FairSpec", properties=["Finishes"], check_deadlock=False)
print(good.outcome.value, "--", good.stats.distinct, "distinct states")

## Where to go next

- [`scratch.ipynb`](/notebooks/index.html?path=scratch.ipynb) — an empty page with setup already written, for a spec of your own.
- [`tla-in-your-browser.ipynb`](/lab/index.html?path=tla-in-your-browser.ipynb) — the tutorial, if you skipped it.
- [Specifying Systems](https://lamport.azurewebsites.net/tla/book.html) — Lamport's book, free, the actual reference for the language.